# 02 Architecture Zoo — Multi-Layer Perceptron (MLP)

**Status:** Complete

**Learning outcome:** Build, train, and probe a multi-layer perceptron on MNIST — understanding universal approximation, depth vs width, and what happens without hidden layers.

## Why This Architecture?

Before MLPs, machine learning was limited to linear models — logistic regression, linear SVMs, simple perceptrons. These models can only carve the input space with a single hyperplane. If the decision boundary is curved, nested, or involves interactions between features, a linear model cannot learn it.

The multi-layer perceptron solves this by stacking layers of linear transformations with nonlinear activation functions in between. Each layer learns a new representation of the input. The hidden layers act as learned feature extractors: they automatically discover the intermediate representations that make the final classification easy.

The **Universal Approximation Theorem** (Cybenko 1989, Hornik 1991) guarantees that a single hidden layer with enough neurons can approximate *any* continuous function on a compact domain to arbitrary precision. This doesn't say how many neurons you need (it could be astronomical), and it doesn't say training will find the right weights — but it tells us that the architecture is not the bottleneck. In practice, deeper networks with moderate width learn more efficiently than shallow-and-wide ones.

## Theory Sketch

### Layers as Function Composition

An MLP is a chain of affine transformations interleaved with element-wise nonlinearities:

$$\mathbf{h}_1 = \sigma(\mathbf{W}_1 \mathbf{x} + \mathbf{b}_1)$$
$$\mathbf{h}_2 = \sigma(\mathbf{W}_2 \mathbf{h}_1 + \mathbf{b}_2)$$
$$\hat{\mathbf{y}} = \mathbf{W}_3 \mathbf{h}_2 + \mathbf{b}_3$$

where $\sigma$ is a nonlinear activation (ReLU, tanh, sigmoid). Without $\sigma$, the composition of linear functions is still linear — multiple layers would collapse into one.

### Depth vs Width

- **Width** (neurons per layer): controls expressiveness at each representation level
- **Depth** (number of layers): controls the compositional complexity of learned features
- Deeper networks can represent certain functions exponentially more efficiently than shallow ones (Telgarsky 2016), but are harder to train (vanishing gradients, which we address in later notebooks)

### Forward and Backward

**Forward pass:** Input flows through each layer, producing activations.

**Backward pass (backpropagation):** Gradients flow from the loss backward through each layer via the chain rule:
$$\frac{\partial L}{\partial \mathbf{W}_l} = \frac{\partial L}{\partial \mathbf{h}_l} \cdot \frac{\partial \mathbf{h}_l}{\partial \mathbf{W}_l}$$

For a ReLU layer $\mathbf{h} = \max(0, \mathbf{W}\mathbf{x} + \mathbf{b})$:
- $\frac{\partial \mathbf{h}}{\partial \mathbf{W}} = \mathbf{x}^T$ (where ReLU is active)
- $\frac{\partial \mathbf{h}}{\partial \mathbf{x}} = \mathbf{W}^T$ (where ReLU is active)

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms

np.random.seed(42)
torch.manual_seed(42)
print("Libraries loaded.")

Libraries loaded.


---
**Understanding Universal Approximation**

**Plain language:** Imagine you have a huge box of Lego bricks. The Universal Approximation Theorem says that with enough bricks (neurons), you can build any shape (function) you want. It does not tell you which bricks to use or how to snap them together — just that the right arrangement exists. A neural network with one hidden layer is like that box: powerful in principle, but you still need a good strategy (training) to find the right arrangement.

**Building intuition:** A ReLU neuron computes $\max(0, w \cdot x + b)$, which is a "hinge" — it's zero on one side and linear on the other. A layer of ReLU neurons creates many hinges at different positions and angles. By combining these hinges (with the output layer weights), you can approximate any piecewise-linear function, and with enough pieces, any continuous function. More neurons = more hinges = finer approximation. Depth helps because composing hinges can create exponentially more "pieces" than adding them.

**Formal statement:** (Cybenko 1989 / Hornik 1991) Let $\sigma$ be any continuous, non-constant, bounded activation function (e.g., sigmoid). For any continuous function $f: [0,1]^d \to \mathbb{R}$ and any $\varepsilon > 0$, there exists $N \in \mathbb{N}$ and weights $\{w_i, b_i, \alpha_i\}$ such that $\left| \sum_{i=1}^{N} \alpha_i \sigma(w_i^T x + b_i) - f(x) \right| < \varepsilon$ for all $x \in [0,1]^d$. Extensions by Leshno et al. (1993) showed this holds for any non-polynomial activation, including ReLU.

---

---
**Understanding Kaiming Initialization**

**Plain language:** Imagine passing a message through a long chain of people. If each person whispers quieter than they heard, the message fades to silence by the end (vanishing). If each person shouts louder, the message becomes deafening noise (exploding). Kaiming initialization is like calibrating each person's volume so the message arrives at the end with the same loudness it started with. It sets the initial weights so that signals neither shrink nor grow as they pass through layers — giving training a healthy starting point.

**Building intuition:** When data flows through a layer with $n$ inputs, each output is a sum of $n$ terms (weights times inputs). If each weight is drawn from a distribution with variance $\sigma^2_w$ and the inputs have variance $\sigma^2_x$, the output variance is $n \cdot \sigma^2_w \cdot \sigma^2_x$. To keep the output variance equal to the input variance, we need $\sigma^2_w = 1/n$. But ReLU zeros out roughly half the activations, effectively halving the surviving signal, so we compensate by doubling the variance: $\sigma^2_w = 2/n$. That is exactly what `np.sqrt(2.0 / input_dim)` does in our NumPyMLP constructor. Without this factor-of-2 correction, activations would shrink by ~$\sqrt{0.5}$ at each layer, causing deep networks to start with near-zero activations.

**Formal statement:** (He et al., 2015) For a layer computing $\mathbf{y} = \mathbf{W}\mathbf{x}$ followed by $\text{ReLU}$, assuming elements of $\mathbf{x}$ are i.i.d. with zero mean: $\text{Var}(y_i) = n_{\text{in}} \cdot \text{Var}(w_{ij}) \cdot E[x_j^2]$. For ReLU, $E[x_j^2] = \frac{1}{2}\text{Var}(x_j)$ (half the pre-activation mass is zeroed). Variance preservation $\text{Var}(y_i) = \text{Var}(x_j)$ requires $\text{Var}(w_{ij}) = \frac{2}{n_{\text{in}}}$, giving $w_{ij} \sim \mathcal{N}(0, \frac{2}{n_{\text{in}}})$. The backward-pass analog requires $\text{Var}(w_{ij}) = \frac{2}{n_{\text{out}}}$. For non-ReLU activations, the factor of 2 is replaced by $1/\text{Var}(\sigma'(z))$; Glorot/Xavier initialization uses $\frac{2}{n_{\text{in}} + n_{\text{out}}}$ (appropriate for linear or tanh activations).

---

## From-Scratch Implementation (NumPy)

A 2-layer MLP with ReLU activation, trained on a small subset of MNIST. We implement both forward and backward passes manually to understand the gradient flow.

In [2]:
# Load MNIST via torchvision, then convert to NumPy for the from-scratch section
_train = datasets.MNIST('data', train=True, download=True, transform=transforms.ToTensor())
_test = datasets.MNIST('data', train=False, transform=transforms.ToTensor())

X_train_full = _train.data.numpy().reshape(-1, 784).astype(np.float32) / 255.0
y_train_full = _train.targets.numpy()
X_test = _test.data.numpy().reshape(-1, 784).astype(np.float32) / 255.0
y_test = _test.targets.numpy()

print(f"Train: {X_train_full.shape}, Test: {X_test.shape}")
print(f"Labels: {np.unique(y_train_full)}")

Train: (60000, 784), Test: (10000, 784)
Labels: [0 1 2 3 4 5 6 7 8 9]


In [3]:
class NumPyMLP:
    """2-layer MLP: input(784) -> hidden(128, ReLU) -> output(10, softmax)."""

    def __init__(self, input_dim=784, hidden_dim=128, output_dim=10):
        # Kaiming init for ReLU
        self.W1 = np.random.randn(input_dim, hidden_dim).astype(np.float32) * np.sqrt(2.0 / input_dim)
        self.b1 = np.zeros(hidden_dim, dtype=np.float32)
        self.W2 = np.random.randn(hidden_dim, output_dim).astype(np.float32) * np.sqrt(2.0 / hidden_dim)
        self.b2 = np.zeros(output_dim, dtype=np.float32)

    def forward(self, X):
        """Forward pass. Cache intermediates for backward."""
        self.X = X                                    # (B, 784)
        self.z1 = X @ self.W1 + self.b1               # (B, 128)
        self.h1 = np.maximum(0, self.z1)               # ReLU
        self.z2 = self.h1 @ self.W2 + self.b2          # (B, 10)
        # Numerically stable softmax
        exp_z = np.exp(self.z2 - self.z2.max(axis=1, keepdims=True))
        self.probs = exp_z / exp_z.sum(axis=1, keepdims=True)  # (B, 10)
        return self.probs

    def backward(self, y):
        """Backward pass. Returns gradients for all parameters."""
        B = y.shape[0]
        # Softmax + cross-entropy gradient: dL/dz2 = probs - one_hot(y)
        one_hot = np.zeros_like(self.probs)
        one_hot[np.arange(B), y] = 1.0
        dz2 = (self.probs - one_hot) / B               # (B, 10)

        # Layer 2 gradients
        self.dW2 = self.h1.T @ dz2                     # (128, 10)
        self.db2 = dz2.sum(axis=0)                      # (10,)

        # Backprop through ReLU
        dh1 = dz2 @ self.W2.T                           # (B, 128)
        dz1 = dh1 * (self.z1 > 0).astype(np.float32)   # ReLU derivative

        # Layer 1 gradients
        self.dW1 = self.X.T @ dz1                       # (784, 128)
        self.db1 = dz1.sum(axis=0)                       # (128,)

    def step(self, lr=0.1):
        """SGD update."""
        self.W1 -= lr * self.dW1
        self.b1 -= lr * self.db1
        self.W2 -= lr * self.dW2
        self.b2 -= lr * self.db2

print("NumPyMLP defined.")

NumPyMLP defined.


---
**Understanding ReLU Activation**

**Plain language:** Imagine a gate that either lets water through or blocks it completely. If the water pressure (input) is positive, the gate opens and passes the water unchanged. If the pressure is zero or negative, the gate slams shut — nothing gets through. ReLU works exactly like this: it keeps positive signals intact and kills negative ones. Older activations like sigmoid were more like a dimmer switch that squishes everything into a narrow range, which caused signals to fade away in deep networks. ReLU's all-or-nothing behavior keeps strong signals strong.

**Building intuition:** ReLU computes $\max(0, z)$ — it's piecewise linear with a kink at zero. This has three practical consequences. First, its gradient is either 1 (active) or 0 (dead), so there's no gradient shrinkage for active neurons — contrast this with sigmoid, where the max gradient is only 0.25. Second, ReLU produces sparse activations: on any given input, roughly half the neurons output exactly zero, which acts as a form of implicit regularization. Third, the "dead neuron" problem: if a neuron's bias drifts so negative that it never activates on any training example, its gradient is permanently zero and it stops learning. In our MNIST MLP, we can count dead neurons by checking per-neuron mean activations — healthy networks keep most neurons alive.

**Formal statement:** The Rectified Linear Unit is defined as $\text{ReLU}(z) = \max(0, z)$ with subgradient $\frac{\partial}{\partial z}\text{ReLU}(z) = \mathbf{1}[z > 0]$. Unlike sigmoid ($\sigma'(z) \leq 0.25$) or tanh ($|\tanh'(z)| \leq 1$ with saturation), ReLU has unit gradient for all $z > 0$, mitigating the vanishing gradient problem (Glorot et al., 2011). A layer of $n$ ReLU neurons partitions input space into at most $2^n$ linear regions (each defined by a different activation pattern), enabling piecewise-linear function approximation. The sparsity of ReLU activations — $P(h_i = 0) \approx 0.5$ for zero-mean pre-activations — provides computational efficiency and implicit regularization analogous to dropout (Glorot & Bengio, 2011).

---

In [4]:
# Train NumPy MLP on a 10k subset for speed, full-batch gradient descent
np.random.seed(42)
X_sub, y_sub = X_train_full[:10000], y_train_full[:10000]

model_np = NumPyMLP()
np_losses, np_accs = [], []

for epoch in range(10):
    probs = model_np.forward(X_sub)
    # Cross-entropy loss
    loss = -np.log(probs[np.arange(len(y_sub)), y_sub] + 1e-8).mean()
    preds = probs.argmax(axis=1)
    acc = (preds == y_sub).mean()
    np_losses.append(loss)
    np_accs.append(acc)

    model_np.backward(y_sub)
    model_np.step(lr=0.5)

    print(f"Epoch {epoch+1:2d}  loss={loss:.4f}  acc={acc:.4f}")

# Verify gradients against PyTorch on a small batch
X_check = torch.tensor(X_sub[:32])
y_check = torch.tensor(y_sub[:32].astype(np.int64))

W1_pt = torch.tensor(model_np.W1.copy(), dtype=torch.float32, requires_grad=True)
b1_pt = torch.tensor(model_np.b1.copy(), dtype=torch.float32, requires_grad=True)
W2_pt = torch.tensor(model_np.W2.copy(), dtype=torch.float32, requires_grad=True)
b2_pt = torch.tensor(model_np.b2.copy(), dtype=torch.float32, requires_grad=True)

h = F.relu(X_check @ W1_pt + b1_pt)
logits = h @ W2_pt + b2_pt
loss_pt = F.cross_entropy(logits, y_check)
loss_pt.backward()

# Compare gradients
probs_check = model_np.forward(X_sub[:32])
model_np.backward(y_sub[:32])

dW1_diff = np.abs(model_np.dW1 - W1_pt.grad.numpy()).max()
dW2_diff = np.abs(model_np.dW2 - W2_pt.grad.numpy()).max()
print(f"\nGradient check — max |dW1 diff|: {dW1_diff:.6f}, max |dW2 diff|: {dW2_diff:.6f}")
assert dW1_diff < 1e-4, f"W1 gradient mismatch: {dW1_diff}"
assert dW2_diff < 1e-4, f"W2 gradient mismatch: {dW2_diff}"
print("NumPy gradients match PyTorch ✓")

Epoch  1  loss=2.4147  acc=0.1247
Epoch  2  loss=1.9697  acc=0.2965
Epoch  3  loss=1.6899  acc=0.5591
Epoch  4  loss=1.4617  acc=0.6086
Epoch  5  loss=1.4351  acc=0.5262
Epoch  6  loss=1.3558  acc=0.6408


Epoch  7  loss=1.1809  acc=0.6323
Epoch  8  loss=1.0300  acc=0.7064
Epoch  9  loss=1.1513  acc=0.6406


Epoch 10  loss=0.9650  acc=0.7001



Gradient check — max |dW1 diff|: 0.000000, max |dW2 diff|: 0.000000
NumPy gradients match PyTorch ✓


---
**Understanding Softmax and Cross-Entropy Loss**

**Plain language:** Imagine a panel of judges each holding up a score card for 10 contestants. The raw scores can be any numbers — some high, some low, some negative. Softmax is like converting those scores into percentages that add up to 100%: the highest score gets the biggest share, and all shares are positive. Cross-entropy loss then measures how surprised we are by the answer: if the model gives 95% to the correct digit, surprise is low (small loss); if it gives only 5%, surprise is high (big loss). Training minimizes this surprise.

**Building intuition:** Softmax converts logits $z_i$ into probabilities via $p_i = e^{z_i} / \sum_j e^{z_j}$. The exponential amplifies differences — a logit gap of 5 becomes a probability ratio of ~150:1. Cross-entropy loss for the true class $k$ is $L = -\log(p_k)$, which has a beautifully simple gradient: $\frac{\partial L}{\partial z_i} = p_i - \mathbf{1}[i=k]$. This means the gradient is just "predicted probability minus actual label" — the same elegant form as logistic regression. No second derivatives of the activation are needed, which makes training stable and fast. The `(probs - one_hot) / B` line in our backward pass implements exactly this.

**Formal statement:** For a $K$-class classification with logits $\mathbf{z} \in \mathbb{R}^K$, the softmax function defines a categorical distribution $p_i = \frac{\exp(z_i)}{\sum_{j=1}^K \exp(z_j)}$. The cross-entropy loss for true class $y$ is $L = -\log p_y = -z_y + \log\sum_{j=1}^K \exp(z_j)$. The gradient $\frac{\partial L}{\partial z_i} = p_i - \mathbf{1}[i = y]$ follows from the identity $\frac{\partial p_i}{\partial z_j} = p_i(\mathbf{1}[i=j] - p_j)$. Cross-entropy is the KL divergence $D_{\text{KL}}(q \| p) + H(q)$ where $q$ is the one-hot target distribution, making it a proper scoring rule (Gneiting & Raftery, 2007). Numerical stability requires the log-sum-exp trick: $\log\sum \exp(z_j) = c + \log\sum\exp(z_j - c)$ with $c = \max_j z_j$.

---

---
**Understanding Backpropagation Through Layers**

**Plain language:** Think of an assembly line where each station modifies the product. If the final product has a defect, the quality inspector traces backward: "Station 3 bent the bracket, but that was because Station 2 sent a warped piece, which was because Station 1 used the wrong material." Backpropagation does exactly this — it traces the "blame" for the loss backward through each layer, telling each set of weights how to adjust.

**Building intuition:** At each layer, we need two things for the backward pass: (1) the local Jacobian — how this layer's output changes with respect to its inputs and weights, and (2) the incoming gradient from the layer above. The chain rule multiplies them together. For ReLU, the local Jacobian is trivially simple: it's an identity for positive pre-activations and zero otherwise. This makes ReLU both computationally cheap and gradient-friendly (no saturation for positive values), which is why it replaced sigmoid/tanh as the default activation.

**Formal statement:** For layer $l$ computing $\mathbf{h}_l = \sigma(\mathbf{W}_l \mathbf{h}_{l-1} + \mathbf{b}_l)$ with loss $L$, let $\delta_l = \frac{\partial L}{\partial \mathbf{z}_l}$ where $\mathbf{z}_l = \mathbf{W}_l \mathbf{h}_{l-1} + \mathbf{b}_l$. Then: $\frac{\partial L}{\partial \mathbf{W}_l} = \mathbf{h}_{l-1}^T \delta_l$, $\frac{\partial L}{\partial \mathbf{b}_l} = \mathbf{1}^T \delta_l$, and $\delta_{l-1} = (\delta_l \mathbf{W}_l^T) \odot \sigma'(\mathbf{z}_{l-1})$, where $\odot$ is element-wise multiplication and $\sigma'$ is the activation derivative. For ReLU, $\sigma'(z) = \mathbf{1}[z > 0]$.

---

## PyTorch Rewrite

Same architecture, now using `nn.Linear`, `nn.ReLU`, and `nn.CrossEntropyLoss`. Train on the full 60k MNIST for 10 epochs with mini-batch SGD.

In [5]:
class MLP(nn.Module):
    """2-layer MLP: 784 -> 128 (ReLU) -> 10."""

    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 128)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = x.view(x.size(0), -1)  # flatten
        x = self.relu(self.fc1(x))
        return self.fc2(x)

model = MLP()
total_params = sum(p.numel() for p in model.parameters())
print(f"MLP parameters: {total_params:,}")
print(f"  fc1: {784*128 + 128:,} (weights + bias)")
print(f"  fc2: {128*10 + 10:,} (weights + bias)")
print(model)

MLP parameters: 101,770
  fc1: 100,480 (weights + bias)
  fc2: 1,290 (weights + bias)
MLP(
  (fc1): Linear(in_features=784, out_features=128, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=128, out_features=10, bias=True)
)


## Training Run

In [6]:
# DataLoaders
torch.manual_seed(42)
train_loader = torch.utils.data.DataLoader(
    datasets.MNIST('data', train=True, transform=transforms.ToTensor()),
    batch_size=128, shuffle=True
)
test_loader = torch.utils.data.DataLoader(
    datasets.MNIST('data', train=False, transform=transforms.ToTensor()),
    batch_size=256, shuffle=False
)

model = MLP()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
criterion = nn.CrossEntropyLoss()

train_losses, train_accs, test_accs = [], [], []

for epoch in range(10):
    # Train
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * X_batch.size(0)
        correct += (logits.argmax(1) == y_batch).sum().item()
        total += X_batch.size(0)

    train_loss = running_loss / total
    train_acc = correct / total
    train_losses.append(train_loss)
    train_accs.append(train_acc)

    # Eval
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            logits = model(X_batch)
            correct += (logits.argmax(1) == y_batch).sum().item()
            total += X_batch.size(0)
    test_acc = correct / total
    test_accs.append(test_acc)

    print(f"Epoch {epoch+1:2d}  train_loss={train_loss:.4f}  "
          f"train_acc={train_acc:.4f}  test_acc={test_acc:.4f}")

# Silent correctness assertion [RAIE v3]
assert test_accs[-1] > 0.95, f"Test accuracy {test_accs[-1]:.4f} below 0.95 threshold"
print(f"\nFinal test accuracy: {test_accs[-1]:.4f} (> 0.95 ✓)")

Epoch  1  train_loss=0.6146  train_acc=0.8387  test_acc=0.9096


Epoch  2  train_loss=0.2961  train_acc=0.9147  test_acc=0.9279


Epoch  3  train_loss=0.2435  train_acc=0.9311  test_acc=0.9347


Epoch  4  train_loss=0.2074  train_acc=0.9419  test_acc=0.9457


Epoch  5  train_loss=0.1804  train_acc=0.9498  test_acc=0.9517


Epoch  6  train_loss=0.1593  train_acc=0.9552  test_acc=0.9564


Epoch  7  train_loss=0.1430  train_acc=0.9594  test_acc=0.9606


Epoch  8  train_loss=0.1293  train_acc=0.9636  test_acc=0.9634


Epoch  9  train_loss=0.1173  train_acc=0.9676  test_acc=0.9657


Epoch 10  train_loss=0.1081  train_acc=0.9701  test_acc=0.9670

Final test accuracy: 0.9670 (> 0.95 ✓)


---
**Understanding Mini-batch SGD with Momentum**

**Plain language:** Imagine rolling a ball down a hilly landscape to find the lowest valley. Pure gradient descent is like placing the ball and letting it slide — it follows the steepest slope at each point, but can get stuck oscillating in narrow ravines. Mini-batches are like checking the slope using only a handful of nearby measurements instead of surveying the entire mountain — faster per step, noisier, but the noise actually helps escape shallow local minima. Momentum is like giving the ball mass: it accumulates speed in consistent directions and dampens oscillations, so it rolls smoothly through ravines instead of bouncing wall to wall.

**Building intuition:** Full-batch gradient descent computes the exact gradient over all 60,000 training examples — accurate but slow (one update per full pass). Stochastic gradient descent (SGD) uses a single example — fast but extremely noisy. Mini-batch SGD (batch size 128 in our code) is the sweet spot: the gradient estimate has variance $\sim \sigma^2 / B$ where $B$ is the batch size, and modern hardware (GPUs) processes a batch almost as fast as a single example due to parallelism. Momentum ($v_t = \mu v_{t-1} + g_t$, then $\theta_t = \theta_{t-1} - \alpha v_t$) adds a "velocity" that accumulates past gradients. With $\mu = 0.9$, the effective step in a consistent direction is amplified by $\sim 1/(1-\mu) = 10\times$, while oscillating directions cancel out. This is why our PyTorch model (`momentum=0.9`) converges much faster than the NumPy model (no momentum).

**Formal statement:** Mini-batch SGD draws a subset $\mathcal{B} \subset \{1, \ldots, N\}$ with $|\mathcal{B}| = B$ and computes $\hat{g} = \frac{1}{B}\sum_{i \in \mathcal{B}} \nabla_\theta L_i(\theta)$, an unbiased estimator of $\nabla_\theta L(\theta)$ with variance $O(\sigma^2 / B)$. Heavy-ball momentum (Polyak, 1964) maintains a velocity $\mathbf{v}_t = \mu \mathbf{v}_{t-1} + \hat{g}_t$ and updates $\theta_{t+1} = \theta_t - \alpha \mathbf{v}_t$. For a quadratic loss with condition number $\kappa$, optimal SGD converges at rate $O(1/\sqrt{T})$, while momentum achieves acceleration to $O(1/T)$ in the deterministic case and reduces oscillation along high-curvature directions by a factor of $(1-\mu)$ (Sutskever et al., 2013). The effective learning rate along persistent gradient directions is $\alpha / (1 - \mu)$; along oscillating directions, cancellation keeps the effective rate near $\alpha$. PyTorch's `SGD(momentum=0.9)` implements Sutskever's formulation: $\mathbf{v}_t = \mu \mathbf{v}_{t-1} + \nabla L(\theta_t)$, $\theta_{t+1} = \theta_t - \alpha \mathbf{v}_t$.

---

In [7]:
# Learning curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(range(1, 11), train_losses, 'b-o', label='Train loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].set_title('Training Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(range(1, 11), train_accs, 'b-o', label='Train acc')
axes[1].plot(range(1, 11), test_accs, 'r-s', label='Test acc')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Train vs Test Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(0.8, 1.0)

plt.tight_layout()
plt.savefig('../assets/mlp_learning_curves.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

/var/folders/sy/gz5gl6d91cbfwd2z85r62rbc0000gn/T/ipykernel_9722/2971333452.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Internal Probing

Weight distribution histograms for each layer — healthy weights should be roughly symmetric and not collapsing to zero or exploding.

In [8]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for ax, (name, param) in zip(axes.flat, model.named_parameters()):
    data = param.detach().numpy().flatten()
    ax.hist(data, bins=50, alpha=0.7, edgecolor='black', linewidth=0.5)
    ax.set_title(f'{name}  shape={list(param.shape)}  '
                 f'mean={data.mean():.4f}  std={data.std():.4f}')
    ax.axvline(0, color='red', linestyle='--', alpha=0.5)
    ax.set_xlabel('Weight value')
    ax.set_ylabel('Count')

plt.suptitle('Weight Distributions After Training', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../assets/mlp_weight_distributions.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

# Also show activation statistics on a test batch
model.eval()
X_probe, _ = next(iter(test_loader))
with torch.no_grad():
    h1 = model.relu(model.fc1(X_probe.view(X_probe.size(0), -1)))
    h1_np = h1.numpy().flatten()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(h1_np, bins=50, alpha=0.7, edgecolor='black', linewidth=0.5)
axes[0].set_title(f'Hidden layer activations (post-ReLU)\n'
                  f'mean={h1_np.mean():.4f}, std={h1_np.std():.4f}, '
                  f'dead={100*(h1_np == 0).mean():.1f}%')
axes[0].set_xlabel('Activation value')

# Per-neuron mean activation — are any neurons dead?
h1_per_neuron = h1.numpy()  # (B, 128)
neuron_means = h1_per_neuron.mean(axis=0)
axes[1].bar(range(128), np.sort(neuron_means)[::-1], alpha=0.7)
axes[1].set_xlabel('Neuron (sorted by mean activation)')
axes[1].set_ylabel('Mean activation')
axes[1].set_title(f'Per-neuron mean activation\n'
                  f'dead neurons (mean<1e-6): {(neuron_means < 1e-6).sum()}/128')
axes[1].axhline(0, color='red', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('../assets/mlp_activations.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

/var/folders/sy/gz5gl6d91cbfwd2z85r62rbc0000gn/T/ipykernel_9722/2268183226.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/var/folders/sy/gz5gl6d91cbfwd2z85r62rbc0000gn/T/ipykernel_9722/2268183226.py:44: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
**Understanding Weight Distribution Probing**

**Plain language:** A doctor checks your vital signs — temperature, blood pressure, heart rate — to tell if you are healthy without needing to know exactly what you ate for breakfast. Weight distribution probing is the same idea for neural networks. By looking at histograms of the weights and activations, we can diagnose problems without re-running training. Healthy weights are spread out symmetrically around zero. If they all cluster near zero, the network has "gone quiet" and is not learning. If they spread wildly, the network is "shouting" and likely unstable.

**Building intuition:** After training, we expect weight distributions to be roughly bell-shaped and centered near zero, with standard deviations that are reasonable for the layer size (typically 0.01-0.5). Warning signs include: (1) bimodal distributions or heavy tails, suggesting the optimizer is stuck oscillating; (2) weights collapsing to near-zero, meaning the layer is not contributing; (3) very large magnitudes, suggesting exploding gradients or a learning rate that is too high. For activations, ReLU outputs should show a spike at zero (dead region) plus a spread of positive values. If the dead fraction exceeds ~80-90%, too many neurons are inactive — this is the "dying ReLU" problem. Per-neuron mean activation plots (like the bar chart above) quickly reveal whether specific neurons are permanently dead.

**Formal statement:** For a trained network with layer $l$ having weight matrix $\mathbf{W}_l \in \mathbb{R}^{n_{\text{out}} \times n_{\text{in}}}$, diagnostic statistics include: (1) $\bar{w}_l = \frac{1}{n}\sum w_{ij}$ (should be $\approx 0$ by symmetry of the loss landscape); (2) $s_l = \sqrt{\frac{1}{n}\sum w_{ij}^2}$ (effective scale, should be $O(1/\sqrt{n_{\text{in}}})$ for Kaiming-initialized networks); (3) the ratio $\|\Delta \mathbf{W}_l\| / \|\mathbf{W}_l\|$ per update (Bottou's rule of thumb: should be $\sim 10^{-3}$). For ReLU activations $\mathbf{h}_l$, the dead neuron fraction $\frac{1}{n_{\text{out}}} \sum_j \mathbf{1}[\bar{h}_{l,j} < \epsilon]$ (averaged over a batch) should remain below ~10% for a well-trained network. Persistent dead neurons indicate the need for lower learning rates, leaky ReLU variants, or re-initialization strategies.

---

## Failure Mode: No Hidden Layers (Linear Classifier)

What happens if we remove the hidden layer entirely? A single linear layer can only learn linear decision boundaries — it should perform noticeably worse on MNIST.

In [9]:
class LinearClassifier(nn.Module):
    """No hidden layers — just input -> output."""

    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(784, 10)

    def forward(self, x):
        return self.fc(x.view(x.size(0), -1))

torch.manual_seed(42)
linear_model = LinearClassifier()
linear_optimizer = torch.optim.SGD(linear_model.parameters(), lr=0.01, momentum=0.9)

linear_test_accs = []
for epoch in range(10):
    linear_model.train()
    for X_batch, y_batch in train_loader:
        linear_optimizer.zero_grad()
        loss = criterion(linear_model(X_batch), y_batch)
        loss.backward()
        linear_optimizer.step()

    linear_model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            correct += (linear_model(X_batch).argmax(1) == y_batch).sum().item()
            total += X_batch.size(0)
    linear_test_accs.append(correct / total)
    print(f"Epoch {epoch+1:2d}  linear_test_acc={linear_test_accs[-1]:.4f}")

print(f"\nComparison after 10 epochs:")
print(f"  MLP (128 hidden):     {test_accs[-1]:.4f} test accuracy")
print(f"  Linear (no hidden):   {linear_test_accs[-1]:.4f} test accuracy")
print(f"  Gap:                  {test_accs[-1] - linear_test_accs[-1]:.4f}")

# The MLP should beat the linear model
assert test_accs[-1] > linear_test_accs[-1], "MLP should outperform linear classifier"

Epoch  1  linear_test_acc=0.8981


Epoch  2  linear_test_acc=0.9092


Epoch  3  linear_test_acc=0.9146


Epoch  4  linear_test_acc=0.9144


Epoch  5  linear_test_acc=0.9176


Epoch  6  linear_test_acc=0.9182


Epoch  7  linear_test_acc=0.9194


Epoch  8  linear_test_acc=0.9181


Epoch  9  linear_test_acc=0.9192


Epoch 10  linear_test_acc=0.9222

Comparison after 10 epochs:
  MLP (128 hidden):     0.9670 test accuracy
  Linear (no hidden):   0.9222 test accuracy
  Gap:                  0.0448


In [10]:
# Side-by-side comparison plot
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(range(1, 11), test_accs, 'b-o', label='MLP (784→128→10)', linewidth=2)
ax.plot(range(1, 11), linear_test_accs, 'r-s', label='Linear (784→10)', linewidth=2)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Test Accuracy', fontsize=12)
ax.set_title('MLP vs Linear Classifier on MNIST', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim(0.85, 1.0)
plt.tight_layout()
plt.savefig('../assets/mlp_vs_linear.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

print("The hidden layer is what gives the MLP its power — without it,")
print("the model can only draw straight lines through 784-dimensional space.")

The hidden layer is what gives the MLP its power — without it,
the model can only draw straight lines through 784-dimensional space.


/var/folders/sy/gz5gl6d91cbfwd2z85r62rbc0000gn/T/ipykernel_9722/3272999005.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
**Understanding Why Depth Matters**

**Plain language:** A linear classifier is like trying to separate groups of people in a crowded room using only a single straight rope. You can separate two big clusters, but if one group is surrounded by the other, no single rope can do it. A hidden layer lets you "reshape the room" first — moving people around so that a straight rope *can* separate them. More hidden layers = more reshaping steps = more complex arrangements become separable.

**Building intuition:** Each neuron in the hidden layer computes a weighted sum of inputs followed by a nonlinearity. This creates a new coordinate system — a learned representation. In the original pixel space, the digit "7" and "1" have overlapping pixel distributions. But in the hidden representation, the network has learned to project them apart. The output layer then simply draws a linear boundary in this learned space. The linear classifier is stuck in pixel space, where many digit pairs are not linearly separable.

**Formal statement:** A single-layer linear classifier computes $\hat{y} = \text{argmax}(\mathbf{W}\mathbf{x} + \mathbf{b})$, defining $K$ half-space regions in $\mathbb{R}^d$ with piecewise-linear boundaries. The expressible decision regions are convex (Minsky & Papert, 1969). An MLP with ReLU hidden layers computes a piecewise-linear function with exponentially more linear regions (Montúfar et al., 2014): a network with $L$ layers of width $n$ can represent $O((\lfloor n/d \rfloor)^{(L-1)d} \cdot n^d)$ linear regions, creating far more complex (including non-convex) decision boundaries.

---

## Structured Interpretation

### Results
- **NumPy MLP** (10k subset, full-batch): accuracy climbs across 10 epochs; gradients verified against PyTorch to < 1e-4 max absolute error
- **PyTorch MLP** (60k train, mini-batch SGD with momentum): test accuracy > 95% after 10 epochs on MNIST
- **Linear classifier**: converges to ~92% test accuracy — a clear ~5% gap vs the MLP
- **Weight distributions**: roughly symmetric around zero after training, no degenerate collapse
- **Activations**: ReLU produces a healthy mix of active and zeroed-out neurons; no fully dead neurons observed

### Findings
- A single hidden layer of 128 neurons is sufficient to reach >95% on MNIST, confirming that modest width handles this task
- The ~5% accuracy gap between MLP and linear classifier demonstrates that MNIST has non-linear structure that requires learned intermediate representations
- Kaiming initialisation keeps weight distributions healthy throughout training — no vanishing or exploding weights observed

### Implications
- MLPs are the foundation: every architecture in this section (CNN, RNN, Transformer, VAE, Diffusion) builds on the same linear-transform-then-activate pattern
- The MLP treats each pixel independently — it has no notion of spatial structure. This is precisely the limitation that CNNs (next notebook) address
- The probing techniques used here (weight histograms, activation statistics, dead neuron counts) will recur throughout the Architecture Zoo

### Considerations
- MNIST is "too easy" — MLPs get >95% because digit shapes have low intrinsic dimensionality. On CIFAR-10 or ImageNet, MLPs perform much worse than CNNs
- Our MLP flattens the 28x28 image, destroying spatial relationships. A permutation of all pixels would give the same result — the network cannot tell that pixel (0,0) is adjacent to pixel (0,1)
- We used vanilla SGD with momentum. Adam or learning rate scheduling could push accuracy higher, but the point here is the architecture, not the optimiser